Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [9]:
# Carregar as questões a serem usadas, cuja lógia está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-04 11:26:47--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9811 (9.6K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]   9.58K  --.-KB/s    in 0s      

2026-04-04 11:26:47 (72.0 MB/s) - ‘questions.ipynb’ saved [9811/9811]



In [10]:
!pip install -q -U google-genai google-auth
# Importar biblioteca pandas, função genai da biblioteca google e função userdata da biblioteca google.colab.
import pandas as pd
from google import genai
from google.colab import userdata

Google Gemini em nuvém, configuração, definição do modelo e execução da consulta.

In [14]:
def markdown_prepare(papel, categoria, contexto, dificuldade, instrucao):
  # Preparação do prompt em Markdown
  prompt_usuario = f"""
  ### Papel:
  {papel}

  ### Área de atuação:
  {categoria}

  ### Contexto:
  {contexto}

  ### Nível de Dificuldade:
  Com valores de 1 a 4, onde 1 indica o mais simples e 4 o mais complexo.
  {dificuldade}

  ### Instrução:
  {instrucao}
  """

  return prompt_usuario

# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# O uso do Secrets é uma alternativa para que chave de API não fique exposta no código.
# Tal chave previamente criada no Google AI Studio.
# Observação: o nome da chave definido no Google AI Studio precisa ser exatamente o mesmo cadastrado no Secrets.
#             inclusive com diferenciação de letra maiúscula e minúscula.
google_api_key = userdata.get('google_api_key')

# O modelo escolhi do para rodar é o Gemini da Google em nuvém preview de 2026.
model_id = 'gemini-3.1-flash-lite-preview'

# Instancia o modelo.
client_ai = genai.Client(api_key=google_api_key)

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gemini_responses = []

# Repetição que percorre todo Dataframe.
for index, row in df_my_questions.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    dificuldade = row['level']
    instrucao = row['turns']

    # Preparação do prompt em Markdown.
    prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

    # Chamada simples para a API da Google em nuvem.
    response = client_ai.models.generate_content(
        model=model_id,
        contents=prompt_usuario,
        config={
            "temperature": 0.1,  # Conservador.
            "max_output_tokens": 1024
        }
    )

    # Armazenar o resultado em uma lista.
    gemini_responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

# Fechar conexão com a IA
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gemini_response = pd.DataFrame(gemini_responses)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.
Questão 35 processada com sucesso.
Questão 36 processada com sucesso.
Questão 37 processada com sucesso.
Questão 38 processada com sucesso.
Questão 39 processada com sucesso.
Questão 40 processada com sucesso.


Realização de perguntas, uma por uma, por API. Em construção. Não execute.

In [ ]:
# Exibir as 5 primeiras linhas de respostas.
df_gemini_response.head()

In [ ]:
# Objetos mais relevantes até aqui.

# DataFrame com perguntas e respostas da linha guia.
df_question_and_guidelines.info()

# Dataframe com as respostas do gemini, com o campo question para relacionar com o dataframe das perguntas.
df_gemini_response.info()

In [7]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# O uso do Secrets é uma alternativa para que chave de API não fique exposta no código.
# Tal chave previamente criada no Google AI Studio.
# Observação: o nome da chave definido no Google AI Studio precisa ser exatamente o mesmo cadastrado no Secrets.
#             inclusive com diferenciação de letra maiúscula e minúscula.
google_api_key = userdata.get('google_api_key')

# Inicializar o cliente da API.
client_ai = genai.Client(api_key=google_api_key)

# O modelo escolhi do para rodar é o Gemini da Google em nuvém ena sua versão mais recente disponível, limitado gratuitamente à 20 requisições.
model_id = 'gemini-3.1-pro-preview'

# Criar uma lista vazia, para guardar as respostas, por questão de performance.
gemini_responses = []

# Repetição que percorre todo Dataframe.
for index, row in df_questions_and_guidelines.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    papel = row['system']
    categoria = row['category']
    contexto = row['statement']
    instrucao = row['turns']

    try:
      # Chamada simples para a API da Google em nuvem.
      response = client_ai.models.generate_content(
        model=model_id,
        contents=prompt_usuario,
        config={
          "temperature": 0.1,  # Conservador.
          "max_output_tokens": 1024
        }
      )
      # Armazenar o resultado em uma lista.
      gemini_responses.append({"question": questao, "response": response})
      print(f"Questão {questao} processada com sucesso.")
    except Exception as e:
      print(f"Erro na questão {questao}: {e}")
      texto_resposta = "Erro no processamento."

# Fechar conexão com a IA
client_ai.close()

# Para melhor visualização, converter as respostas em um Dataframe.
df_gemini_response = pd.DataFrame(gemini_responses)

NameError: name 'Client' is not defined